In [10]:
import numpy as np
import flopy
import os
from subprocess import Popen, PIPE

In [2]:
# folder containing original model files
org_d = os.path.join('..', "results", '3d_injection_modeling_PMW-6')

# a dir to hold a copy of the org model files
tmp_d = os.path.join('pyemu', 'PMW-6-constantK',  'PMW-6_mf6')

#template dir with PEST files 
template_d = os.path.join('pyemu', 'PMW-6-constantK', 'PMW-6_template')

In [6]:
#loading MODFLOW files
sim = flopy.mf6.MFSimulation.load(sim_ws=tmp_d)
gwf = sim.get_model()

ncol = gwf.modelgrid.ncol
nrow = gwf.modelgrid.nrow


loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package ic...
    loading package npf...
    loading package sto...
    loading package chd...
    loading package wel...
    loading package oc...
    loading package obs...
  loading solution package mlac-model...


In [9]:
#creating tabular file for plotting results in QGIS
def create_grid_file(nrow, ncol, hk_values, filename):
    with open(filename, 'w') as f:
        f.write("row column hk\n")
        idx = 0
        for row in range(1, nrow + 1):
            for col in range(1, ncol + 1):
                f.write(f"{row} {col} {hk_values[row-1][col-1]}\n")
                idx += 1

filename = 'k_layer1.txt'
path_filename = os.path.join('pyemu', 'PMW-6-constantK', 'PMW-6_template', filename)

hk = np.loadtxt(path_filename)
create_grid_file(nrow, ncol, hk, filename=path_filename[:-4]+'_tabular.txt')

In [38]:
#running mf62gis executable for generation of GIS file with dis and npf info
layer = '1'
mf62gis_inputs = [os.path.join(template_d, 'MLAC-model.dis.grb'), 
                        layer, 
                        '',  #ENTER
                        f'../results/visualisation/model_layer{layer}', 
                        os.path.join(template_d, f'k_layer{layer}_tabular.txt'), 
                        'y', 
                        'r', 
                        '-999', 
                        'r'
                        ]

mf62gis_d = os.path.join('supporting_executables', 'mf62gis.exe')

process = Popen(mf62gis_d, stdin=PIPE, stdout=PIPE)
for entry in mf62gis_inputs:
    process.stdin.write((entry + '\n').encode())
process.stdin.close()
process.wait()

print(process.stdout.read().decode())

 
 Program MF62GIS writes a BLN file and MIF/MID files for a MODFLOW6 model.
 
 Enter name of MODFLOW6 binary grid file:  - file pyemu\PMW-6-constantK\PMW-6_template\MLAC-model.dis.grb read ok.
 
 Enter layer number of interest: 
 Enter name for BLN file (<Enter> if none): 
 Enter filename base for MIF/MID files (<Enter> if none): 
 Enter name of tabular data file (<Enter> if none): 
 The following columns have been detected in the table file.
 Indicate whether you would like pertinent data transferred to MIF/MID file.
 
    Data in column labelled "hk"?  [y/n]:     Does column contain integer or real data? [i/r]:     Enter value for missing data:     Add or replace values for duplicated cells [a/r]: 
 - reading tabular data file...
 - 150544 lines of data read from file pyemu\PMW-6-constantK\PMW-6_template\k_layer1_tabular.txt.
 - file ../results/visualisation/model_layer1.mif written ok.
 - file ../results/visualisation/model_layer1.mid written ok.



In [39]:
#running mf6dep2csv executable for generation of csv from .hds
#basically it will be a csv file with info of each cell and recorded heads
#for all time steps. this is a easier way to construct a csv for tabular data file generation
layer = '1'
mf6dep2csv_inputs = [os.path.join(template_d, 'MLAC-model.dis.grb'), 
                        os.path.join(template_d, 'MLAC-model.hds'), 
                        f'../results/visualisation/layer{layer}_heads.csv',  
                        'o', 
                        layer, 
                        'a', 
                        ]

mf6dep2csv_d = os.path.join('supporting_executables', 'mf6dep2csv.exe')

process = Popen(mf6dep2csv_d, stdin=PIPE, stdout=PIPE)
for entry in mf6dep2csv_inputs:
    process.stdin.write((entry + '\n').encode())
process.stdin.close()
process.wait()

print(process.stdout.read().decode())

 
 Program MF6DEP2CSV records MODFLOW6-calculated system states in CSV format.
 
 Enter name of MODFLOW6 binary grid file:  - file pyemu\PMW-6-constantK\PMW-6_template\MLAC-model.dis.grb read ok.
 
 Enter binary MF6-generated dependent variable file: 
 Enter name for CSV output file:  Record "HEAD" data for all model layers or just one? [a/o]:  Record "HEAD" data for which layer?  Record "HEAD" data for all model output times or just one? [a/o]:  - pre-reading file pyemu\PMW-6-constantK\PMW-6_template\MLAC-model.hds...
 - reading file pyemu\PMW-6-constantK\PMW-6_template\MLAC-model.hds...
 - file pyemu\PMW-6-constantK\PMW-6_template\MLAC-model.hds read ok.
 - writing file ../results/visualisation/layer1_heads.csv...
 - file ../results/visualisation/layer1_heads.csv written ok.



In [ ]:
#create a tabular data file with hk, heads for all timesteps